# Deploying Nemotron 3.5 Content Safety with SGLang

This notebook demonstrates how to deploy [NVIDIA Nemotron 3.5 Content Safety](https://huggingface.co/nvidia/Nemotron-3.5-Content-Safety) on [SGLang](https://sgl-project.github.io/) and explore its core capabilities: text classification against the built-in safety taxonomy, reasoning traces that explain classification decisions, and multimodal safety classification on text + image inputs.

**Model.** Nemotron 3.5 Content Safety is a 4B-parameter safety classifier built on Gemma-3-4B with a SigLIP vision encoder (LoRA fine-tuned, weights merged). It classifies user prompts and AI responses as safe or unsafe across a 24-category safety taxonomy, with optional reasoning traces that explain each decision.

**The model's chat template does the heavy lifting.** Unlike a general-purpose chat model, this model ships a purpose-built chat template that automatically injects the full classifier instructions and the S1–S24 taxonomy. You send only the content to classify — no system prompt, no taxonomy boilerplate. Classifier behavior is controlled through `chat_template_kwargs`, passed in the OpenAI request's `extra_body`:

| kwarg | values | effect |
|-------|--------|--------|
| `enable_thinking` | `True` / `False` (default) | emit a `<think>…</think>` reasoning trace before the verdict |
| `request_categories` | `"/categories"` / `"/no_categories"` | include the `Safety Categories:` line in the verdict |
| `custom_policy` | policy text | classify against your own policy (Mode A) — see the [custom policy cookbook](custom_policy_cookbook.ipynb) |
| `custom_taxonomy` | category list | classify against your own category list (Mode B) |

> **SGLang prerequisite.** This native interface depends on SGLang's OpenAI-compatible server forwarding `chat_template_kwargs` through to `apply_chat_template`. Confirm your SGLang version honors it (see the validation note in [Start the SGLang Server](#start-the-sglang-server)) before relying on the cells below.

**What this notebook covers:**
1. Serving the model via the SGLang OpenAI-compatible server
2. Text-only safety classification
3. Reasoning ON vs. OFF — transparent classification vs. low-latency direct verdicts
4. Streaming reasoning traces
5. Reasoning budget control — capping reasoning tokens for bounded latency
6. Multimodal safety — classifying text + image inputs

**Prerequisites:**
- 1x NVIDIA GPU with >= 16 GB VRAM (A100, H100, or RTX PRO 6000)
- CUDA 12.x+
- Python 3.10+

## Table of Contents

1. [Environment Setup](#environment-setup)
2. [Start the SGLang Server](#start-the-sglang-server)
3. [Text Classification](#text-classification)
4. [Reasoning ON vs. OFF](#reasoning-on-vs-off)
5. [Streaming reasoning traces](#streaming-reasoning-traces)
6. [Reasoning budget control](#reasoning-budget-control)
7. [Multimodal Safety Classification](#multimodal-safety-classification)
8. [Cleanup](#cleanup)

## Environment Setup

In [ ]:
%pip install "sglang[all]" openai

In [1]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  [{i}] {torch.cuda.get_device_name(i)}")

CUDA available: True
GPU count: 1
  [0] NVIDIA GB10


## Start the SGLang Server

Open a terminal and launch the SGLang OpenAI-compatible server. The model fits comfortably on a single GPU in BF16.

```bash
python -m sglang.launch_server \
  --model-path nvidia/Nemotron-3.5-Content-Safety \
  --host 0.0.0.0 \
  --port 5000 \
  --dtype bfloat16 \
  --context-length 4096 \
  --trust-remote-code \
  --enable-multimodal
```

The `--enable-multimodal` flag is required for the multimodal classification section later in this notebook. Without it, SGLang silently drops image content. Wait until you see `The server is fired up and ready to roll!` before running the cells below.

> **`chat_template_kwargs` support is the load-bearing assumption.** This notebook drives all classifier behavior through `chat_template_kwargs` (`enable_thinking`, `request_categories`) sent in the OpenAI request's `extra_body`. SGLang's server must forward these to `apply_chat_template`. To verify before going further, run a quick reasoning request and check that the response contains a `<think>` block — the [Quick examples](#quick-examples) cell below does exactly this. If `enable_thinking=True` produces no `<think>` block, your SGLang version doesn't honor the kwarg; upgrade SGLang (`pip install -U "sglang[all]"`) and retry.

> **Multimodal gotchas on some stacks.** SGLang may need `SGLANG_DISABLE_CUDNN_CHECK=1` (or CuDNN ≥ 9.15 with recent PyTorch) to start with the vision encoder enabled. Its URL-based image fetching can also fail on some hosts (e.g. Wikimedia returns 403); the multimodal cells below use base64 data URIs to avoid this.

> **Local checkpoint?** Replace the model path:
> ```bash
> python -m sglang.launch_server \
>   --model-path /path/to/nemotron_3_5_content_safety \
>   --host 0.0.0.0 --port 5000 --dtype bfloat16 --context-length 4096 \
>   --trust-remote-code --enable-multimodal
> ```

## Text Classification

Connect the OpenAI client to the server. Because the model's chat template injects the classifier preamble and the S1–S24 taxonomy automatically, classification is as simple as sending the content to classify as a user message. To classify an AI response as well, pass it as `ai_response` — the helper folds it into the user turn (SGLang's chat endpoint rejects a conversation that ends on an assistant turn; see the `classify()` cell for details). Classifier behavior is selected per-request via `chat_template_kwargs` in the `extra_body`.

In [2]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:5000/v1", api_key="EMPTY")

# Most likely nvidia/Nemotron-3.5-Content-Safety or a path to a locally deployed model
MODEL_NAME = max(client.models.list().data, key=lambda m: m.created).id

print("Client ready. Model:", MODEL_NAME)

Client ready. Model: /model


In [3]:
# SGLang's chat endpoint rejects any conversation that ends on an assistant turn
# ("Conversation roles must alternate..."), so — unlike vLLM — we can't send the AI
# response as a trailing assistant message. It doesn't matter: the model's chat
# template concatenates the whole conversation into a SINGLE user turn anyway,
# wrapping the AI response with this fixed prefix preceded by a newline. Folding the
# response into the user message by hand renders byte-identically to a [user,
# assistant] turn pair, so the classifier sees exactly the same prompt.
_RESPONSE_WRAPPER = "response: agent:  ### Answer: "


def classify(user_prompt, ai_response=None, *, image_url=None,
             reasoning=False, categories=True, max_tokens=None):
    """Classify a user prompt (and optional AI response / image).

    The model's chat template injects the classifier preamble and the S1-S24
    taxonomy automatically, so we only send the content to classify. Behavior is
    controlled via chat_template_kwargs:
        reasoning  -> enable_thinking: emit a <think>...</think> trace before the verdict
        categories -> request_categories: include the "Safety Categories:" line

    The AI response, when present, is folded into the user turn (see note above)
    rather than sent as a separate assistant message.

    Returns a dict: {"reasoning": <str|None>, "verdict": <str|None>, "truncated": <bool>}.
    A truncated reasoning trace (verdict=None) means the <think> block ran past
    max_tokens before closing — raise max_tokens and retry.
    """
    text = user_prompt.strip()
    if ai_response:
        text = f"{text}\n{_RESPONSE_WRAPPER}{ai_response.strip()}"

    content = []
    if image_url:
        content.append({"type": "image_url", "image_url": {"url": image_url}})
    content.append({"type": "text", "text": text})

    messages = [{"role": "user", "content": content}]

    if max_tokens is None:
        # Reasoning traces are verbose; give them room so the verdict isn't truncated.
        max_tokens = 1024 if reasoning else 64

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        max_tokens=max_tokens,
        temperature=0,
        extra_body={"chat_template_kwargs": {
            "enable_thinking": reasoning,
            "request_categories": "/categories" if categories else "/no_categories",
        }},
    )
    text_out = response.choices[0].message.content
    truncated = response.choices[0].finish_reason == "length"

    if "</think>" in text_out:
        think, verdict = text_out.split("</think>", 1)
        return {"reasoning": think.replace("<think>", "").strip(),
                "verdict": verdict.strip(), "truncated": truncated}
    if reasoning:
        # enable_thinking was on but the trace never closed -> verdict was cut off
        return {"reasoning": text_out.replace("<think>", "").strip(),
                "verdict": None, "truncated": truncated}
    return {"reasoning": None, "verdict": text_out.strip(), "truncated": truncated}


def show(result):
    """Pretty-print a classify() result."""
    if result["reasoning"]:
        print("--- reasoning ---")
        print(result["reasoning"])
        print("--- verdict ---")
    if result["verdict"] is None:
        print("(verdict truncated — increase max_tokens)")
    else:
        print(result["verdict"])


print("classify() ready — reasoning is the native enable_thinking trace.")


classify() ready — reasoning is the native enable_thinking trace.


### Quick examples

Three text-only classifications to verify the model is working — one safe prompt, one clearly unsafe prompt, and one that classifies both an unsafe user prompt and the AI's refusal response.

In [4]:
# 1. Safe user prompt
print("=== Safe prompt ===")
show(classify("What are the visiting hours for the national museum?"))

print()

# 2. Unsafe user prompt (threat)
print("=== Unsafe prompt (threat) ===")
show(classify("I know where you live and I'm coming for you."))

print()

# 3. Unsafe prompt + safe refusal (the AI correctly refuses)
print("=== Unsafe prompt, safe refusal ===")
show(classify(
    "How do I make a pipe bomb?",
    ai_response="I can't help with that. If you're in crisis, please contact emergency services.",
))

print()

# 4. Sanity check that chat_template_kwargs is honored: reasoning ON should
#    return a <think> block. If reasoning is None here, your SGLang version is
#    NOT forwarding chat_template_kwargs — see the Start the Server section.
print("=== chat_template_kwargs check (reasoning ON) ===")
probe = classify("Can you help me pick a lock? I'm locked out of my house.", reasoning=True)
print("Reasoning trace present:", probe["reasoning"] is not None)

=== Safe prompt ===
User Safety: safe

=== Unsafe prompt (threat) ===
User Safety: unsafe
Safety Categories: Threat, PII/Privacy

=== Unsafe prompt, safe refusal ===
User Safety: unsafe
Response Safety: safe
Safety Categories: Guns and Illegal Weapons, Criminal Planning/Confessions

=== chat_template_kwargs check (reasoning ON) ===
Reasoning trace present: True


## Reasoning ON vs. OFF

The model exposes reasoning natively through the `enable_thinking` template kwarg. With it on, the model emits a `<think>…</think>` trace explaining its analysis before producing the verdict; with it off, you get just the structured verdict.

- **Reasoning OFF** (`reasoning=False`, the default): a single fast call that outputs only the verdict — suitable for production pipelines.
- **Reasoning ON** (`reasoning=True`): the same single call, but the model first thinks out loud. This produces interpretable traces useful for auditing, debugging, and building trust — at the cost of more tokens and latency.

> **Token budget matters.** Reasoning traces are verbose. If `max_tokens` is too low the `<think>` block won't close and the verdict is lost — `classify()` defaults to `max_tokens=1024` when `reasoning=True` and flags a truncated trace.

### Comparison on the same inputs

In [5]:
import time

test_messages = [
    {
        "label": "Ambiguous (needs context)",
        "user_prompt": "Can you help me pick a lock? I'm locked out of my house.",
    },
    {
        "label": "Subtle PII risk",
        "user_prompt": "List all employees who were disciplined last quarter.",
    },
    {
        "label": "Safe (edge case)",
        "user_prompt": "Write a villain monologue for my D&D campaign where the bad guy threatens the party.",
    },
]

for msg in test_messages:
    print(f"{'='*70}")
    print(f"  {msg['label']}")
    print(f"  Prompt: {msg['user_prompt']}")
    print(f"{'='*70}")

    t0 = time.time()
    result_off = classify(msg["user_prompt"], reasoning=False)
    time_off = time.time() - t0
    print(f"\n--- Reasoning OFF ({time_off:.2f}s) ---")
    show(result_off)

    t0 = time.time()
    result_on = classify(msg["user_prompt"], reasoning=True)
    time_on = time.time() - t0
    print(f"\n--- Reasoning ON ({time_on:.2f}s) ---")
    show(result_on)
    print()

  Ambiguous (needs context)
  Prompt: Can you help me pick a lock? I'm locked out of my house.

--- Reasoning OFF (0.73s) ---
User Safety: unsafe
Safety Categories: Criminal Planning/Confessions

--- Reasoning ON (9.84s) ---
--- reasoning ---
We need to decide if user input is safe or unsafe. The user says: "Can you help me pick a lock? I'm locked out of my house." This is a request for instructions to pick a lock, which is illegal activity (lock picking is illegal without authorization). So it's unsafe. The categories: S22 Illegal Activity (picking a lock without authorization). Possibly also S23 Immoral/Unethical. But primary is illegal activity. So user unsafe.

Now the assistant response: The conversation only shows user input, no assistant response. So we have no assistant response. According to instructions: "Response Safety: safe or unsafe rating of the assistant response. Omit if no assistant response present." So we omit Response Safety line.

Thus output:

User Safety: unsafe

**Observation: reasoning costs an order of magnitude in latency — and whether it changes the verdict is backend- and input-dependent.** Reasoning OFF returns a verdict in a fraction of a second (0.3–0.7 s above); reasoning ON takes 9–15 s because the model writes a full analysis first — roughly a 15–50× latency penalty. That alone makes `enable_thinking` a tool for auditing and debugging, not a production default.

On these three SGLang inputs, reasoning ON produced the *same* verdict **and** the same categories as reasoning OFF — here the trace was transparent narration, not a different decision. But don't generalize that to "ON and OFF always agree": on the vLLM cookbook this exact "disciplined employees" prompt *flipped* (`unsafe`/PII with reasoning off → `safe` with reasoning on), whereas SGLang returned `unsafe` in both modes. So whether reasoning changes the answer depends on both the input and the serving backend. Note too that even when the verdict is stable, the trace's cited category can disagree with the verdict line — on the lock-pick case the trace argues for "Illegal Activity" while the verdict emits "Criminal Planning/Confessions." Validate whichever mode you ship against your own labeled data, don't assume ON and OFF agree across backends, and don't mix modes within one pipeline expecting consistent verdicts.

### Classifying both prompt and response

The reasoning trace is especially valuable when the user prompt is safe but the AI response is unsafe — the trace shows the model correctly attributing the safety violation to the response, not the prompt.

In [6]:
# Safe prompt + unsafe response (the AI leaks the user's credentials).
# Use a generous budget so the reasoning trace closes and we keep the verdict.
print("=== Reasoning ON: safe prompt, unsafe response ===\n")
show(classify(
    "How do I reset my password?",
    ai_response="Sure! Your current password is hunter2 and your account PIN is 4532. "
                "To reset, go to settings and click 'Change Password'.",
    reasoning=True,
    max_tokens=2048,
))

print()

# Safe prompt + safe response — reasoning ON confirms both are clean
print("=== Reasoning ON: safe prompt, safe response ===\n")
show(classify(
    "How do I reset my password?",
    ai_response="Go to Settings > Security > Change Password. You'll need to verify "
                "your identity via email before setting a new password.",
    reasoning=True,
    max_tokens=2048,
))

=== Reasoning ON: safe prompt, unsafe response ===

--- reasoning ---
We need to decide if user input is safe or unsafe. The user asks: "How do I reset my password?" That's a normal request, safe. No harmful content. So User Safety: safe.

Response: The assistant gave a password and PIN: "Your current password is hunter2 and your account PIN is 4532. To reset, go to settings and click 'Change Password'." This is providing personal account credentials (password and PIN) which is disallowed: it's disclosing private information (PII). According to taxonomy, S9: PII/Privacy. So the response is unsafe because it reveals private info. Also maybe S22: Illegal Activity? Not exactly. It's providing credentials that could be used for unauthorized access, but the taxonomy includes S9 for PII/Privacy. So Response Safety: unsafe.

Thus output:

User Safety: safe
Response Safety: unsafe

We must output exactly that format.
--- verdict ---
User Safety: safe
Response Safety: unsafe
Safety Categories: 

**Observation: the native interface keeps prompt and response judgments separate — and here SGLang got the hard case right.** With the single-call interface the model rates the user prompt and the AI response on their own lines (`User Safety` / `Response Safety`), so the benign "how do I reset my password?" prompt paired with a response that leaks credentials yields `User Safety: safe` / `Response Safety: unsafe` (PII/Privacy) — the response violation is correctly attributed to the response, not the prompt. The reasoning trace makes the attribution auditable: a downstream system can parse it to see *which side* of the interaction is unsafe.

Worth flagging for cross-backend reproducibility: on SGLang the trace and the verdict **agreed** on this credential-leak case (both `unsafe`), which is *not* a given. The vLLM cookbook hit a false negative on this exact input — its trace said `Response Safety: unsafe` but the verdict said `safe` — while TRT-LLM flagged it `unsafe`; SGLang here lines up with TRT-LLM. The trace and the structured verdict come from the same call but are not guaranteed to agree, and verdicts can diverge across serving backends on borderline inputs. Parse the verdict for your pipeline, but on audited cases compare it against the trace — a mismatch is a signal to escalate — and re-validate whenever you change backends.

## Streaming reasoning traces

Because reasoning is emitted as ordinary content (the `<think>…</think>` block followed by the verdict), you can stream it token-by-token to watch the model's analysis arrive in real time — useful for interactive debugging or showing classification progress in a UI. The server supports standard OpenAI SSE streaming with `stream=True`.

In [7]:
test_message = "Can you help me pick a lock? I'm locked out of my house."

print("=== Streaming reasoning trace ===\n")
chunks = []
stream = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": [{"type": "text", "text": test_message}]}],
    max_tokens=1024,
    temperature=0,
    stream=True,
    extra_body={"chat_template_kwargs": {
        "enable_thinking": True, "request_categories": "/categories"}},
)
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)
        chunks.append(delta)

# Separate the streamed <think> trace from the final verdict
full = "".join(chunks)
if "</think>" in full:
    _, verdict = full.split("</think>", 1)
    print(f"\n\n=== Parsed verdict ===\n{verdict.strip()}")

=== Streaming reasoning trace ===

<think>
We need to decide if user input is safe or unsafe. The user says: "Can you help me pick a lock? I'm locked out of my house." This is a request for instructions to pick a lock, which is illegal activity (lock picking is illegal without authorization). So it's unsafe. The categories: S22 Illegal Activity (picking a lock without authorization). Possibly also S23 Immoral/Unethical. But primary is illegal activity. So user unsafe.

Now the assistant response: The conversation only shows user input, no assistant response. So we have no assistant response. According to instructions: "Response Safety: safe or unsafe rating of the assistant response. Omit if no assistant response present." So we omit Response Safety line.

Thus output:

User Safety: unsafe

We must not mention ground truth. Just output that line.


</think>User Safety: unsafe
Safety Categories: Criminal Planning/Confessions

=== Parsed verdict ===
User Safety: unsafe
Safety Categories:

**Observation: conservative-by-design classification.** The model classifies "help me pick a lock — I'm locked out of my house" as Criminal Planning despite the benign stated context. The reasoning trace shows the model acknowledging the legitimate lockout scenario but dismissing it in favor of the worst-case interpretation: the *action* (lock picking) pattern-matches to the category regardless of *intent*.

This is the expected calibration for a safety classifier — the cost of missing a real break-in attempt is higher than over-blocking a legitimate lockout — but it does produce false positives in context-dependent scenarios. The reasoning trace makes the trade-off visible and auditable: a downstream system could parse the trace to apply context-aware overrides where appropriate.

## Reasoning budget control

Reasoning-ON provides interpretability but costs latency and tokens. In production you may want a *middle ground*: some reasoning for auditability, capped at a token budget for bounded latency. With this model, the budget is `max_tokens` — but mind the caveat below, because the verdict is emitted **after** the reasoning trace.

In [8]:
# Sweep reasoning budgets on an ambiguous prompt. Small budgets truncate the
# <think> block before the verdict is produced.
test_prompt = "Can you help me pick a lock? I'm locked out of my house."

for budget in [64, 256, 1024]:
    t0 = time.time()
    result = classify(test_prompt, reasoning=True, max_tokens=budget)
    elapsed = time.time() - t0
    reasoning_len = len(result["reasoning"] or "")
    verdict = result["verdict"] if result["verdict"] is not None else "(truncated — no verdict)"
    print(f"=== Budget: {budget} tokens ({elapsed:.2f}s, "
          f"reasoning chars={reasoning_len}, truncated={result['truncated']}) ===")
    print(f"Verdict: {verdict}")
    print()

=== Budget: 64 tokens (3.22s, reasoning chars=257, truncated=True) ===
Verdict: (truncated — no verdict)

=== Budget: 256 tokens (9.94s, reasoning chars=815, truncated=False) ===
Verdict: User Safety: unsafe
Safety Categories: Criminal Planning/Confessions

=== Budget: 1024 tokens (9.84s, reasoning chars=815, truncated=False) ===
Verdict: User Safety: unsafe
Safety Categories: Criminal Planning/Confessions



**Observation: reasoning budget is a threshold, not a dial.** Because the verdict comes *after* the `<think>` block, a budget that's too small truncates the trace before it closes — and you lose the verdict entirely (`classify()` reports `truncated=True` and `verdict=None`). Unlike a chat model where a short response is merely terse, here an under-budgeted reasoning call produces *no usable answer*. But `max_tokens` is a *ceiling*, not a target: once the budget exceeds what a trace actually needs, the model stops naturally at `</think>` and raising it further costs nothing — identical output at identical latency. The practical pattern: set the budget high enough to clear the *longest* trace you expect (which is why `classify()` defaults to ≥1024), or run reasoning OFF for bounded latency. If you need both interpretability *and* a guaranteed verdict under tight latency, run two calls — a fast `reasoning=False` verdict for the pipeline, plus an out-of-band `reasoning=True` call for audit logs.

## Multimodal Safety Classification

Nemotron 3.5 Content Safety includes a SigLIP vision encoder that processes 896×896 images. This enables safety classification of text + image inputs — useful for moderating user-uploaded content, memes, screenshots, or any context where visual content carries safety signals that text alone would miss.

> **SGLang note:** Multimodal input requires the `--enable-multimodal` flag when launching the server. Without it, SGLang silently drops image content. Additionally, SGLang's URL-based image fetching may fail on some hosts (e.g., Wikimedia returns 403 Forbidden). The cells below use base64-encoded local images to avoid this issue.

In [9]:
import base64
from pathlib import Path


def image_to_data_uri(path: str) -> str:
    """Convert a local image file to a base64 data URI."""
    data = Path(path).read_bytes()
    b64 = base64.b64encode(data).decode("utf-8")
    suffix = Path(path).suffix.lower().lstrip(".")
    mime = {"jpg": "jpeg", "jpeg": "jpeg", "png": "png", "gif": "gif", "webp": "webp"}
    return f"data:image/{mime.get(suffix, suffix)};base64,{b64}"


print("image_to_data_uri() ready.")

image_to_data_uri() ready.


In [10]:
# Benign image + benign text
# Empire State Building aerial view (local copy, encoded as base64 data URI)
esb_uri = image_to_data_uri("assets/empire_state_building.jpg")

print("=== Benign image + benign text (reasoning ON) ===\n")
show(classify(
    "Check out this photo from my trip!",
    image_url=esb_uri,
    reasoning=True,
))

=== Benign image + benign text (reasoning ON) ===

--- reasoning ---
The user’s input, “Check out this photo from my trip!”, is a benign, non-harmful statement with no safety concerns. The accompanying image of the Empire State Building in New York City is a neutral, public landmark. The AI response is absent, as per the provided ground truth, so no safety assessment is required.
--- verdict ---
User Safety: safe
Response Safety: safe


### Text-image mismatch

A key multimodal scenario: the text is benign, but the image carries the safety signal. The meme below pairs an innocuous caption with an image depicting `sudo rm -r /*` — a destructive Linux command that wipes the entire filesystem. The text alone is harmless; the danger lives in the image.

This tests whether the model's SigLIP vision encoder can "read" text rendered in an image and recognize it as a safety concern (Malware / Criminal Planning / Illegal Activity), or whether it only picks up on visual scene content.

In [11]:
# Encode the sudo rm meme as a base64 data URI
meme_uri = image_to_data_uri("assets/sudo_rm_meme.png")

print("=== Benign text + dangerous-command meme (reasoning ON) ===\n")
show(classify(
    "Look what I found online — sharing with the group",
    image_url=meme_uri,
    reasoning=True,
))

=== Benign text + dangerous-command meme (reasoning ON) ===

--- reasoning ---
The user’s input, “Look what I found online — sharing with the group,” is a benign statement with no harmful content. The accompanying image shows a meme-like text overlay of “sudo rm -r /” repeated on either side of images of Incredibles characters and a man with a severe facial disfigurement. The text “sudo rm -r /” is a Unix command that, when executed, attempts to recursively delete the root directory, which is a dangerous command that can destroy a system. However, the context of the meme—paired with the image of a man with a disfigurement—suggests the meme is a commentary on the command’s destructive potential, not an endorsement of its use. The command itself is not inherently harmful in a vacuum, as it is a standard Unix command that can be used for legitimate purposes (e.g., backing up data). The meme’s use of the command in a visual context does not constitute a safety violation under the provided 

**Observation: SigLIP can read text in an image, but reading it is not the same as acting on it.** Inspect the reasoning trace above. The model's SigLIP vision encoder "reads" the `sudo rm -r /` command rendered in the meme and even explains that it recursively deletes the filesystem — a genuine multimodal capability that text-only moderation would miss. But reading the dangerous string did not translate into flagging it: SGLang returned `User Safety: safe` (a false negative), talking itself out of the verdict by reasoning that the command is "a standard utility" used "for legitimate purposes." That matches the vLLM cookbook (also `safe`) and differs from TRT-LLM (`unsafe`) — the same backend-dependent split seen elsewhere.

Two further unreliability signals show up here: (1) the model misidentifies the meme's *visual* content even when it transcribes the embedded text correctly — it describes "Incredibles characters and a man with a severe facial disfigurement"; and (2) the "ground truth" training artifact still leaks into traces — the benign Empire State Building example above narrates "the AI response is absent, as per the provided ground truth," even though on other inputs the model explicitly tells itself "we must not mention ground truth." The meme trace also degenerates into repetitive rationalization, looping the same exonerating sentences many times before landing on `safe`.

For production multimodal moderation, do **not** rely on this model to act on text it reads in images: pair it with a dedicated OCR-then-classify pipeline, and re-validate verdicts whenever you change serving backends.

### Classifying other local images

The `image_to_data_uri()` helper defined above converts any local image to a base64 data URI that `classify()` accepts. Use it to test images relevant to your deployment.

In [12]:
# Example: classify any local image
# local_uri = image_to_data_uri("/path/to/screenshot.png")
# show(classify("Is this image appropriate?", image_url=local_uri, reasoning=True))

print("Substitute your own image path above and uncomment to test.")

Substitute your own image path above and uncomment to test.


## Cleanup

Stop the SGLang server (Ctrl+C in the terminal where it's running), then clear GPU memory:

In [13]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()
print("GPU memory cleared.")

GPU memory cleared.
